# MCP Client — JSON-RPC 2.0 for Snowflake Managed MCP

> A Python client for interacting with Snowflake's Managed MCP Server via the standard MCP protocol

In [ ]:
#| default_exp mcp_client

In [ ]:
#| export
import json
from collections.abc import Callable
from html import escape as _html_escape

import requests
import pandas as pd
from mcp_ski_resort.core import (
    default_session, get_headers,
    extract_text, try_parse_json,
    parse_agent_response,
    result_set_to_dataframe,
)

## The MCP Protocol

Snowflake's Managed MCP Server speaks [JSON-RPC 2.0](https://www.jsonrpc.org/specification) with two key methods:

- **`tools/list`** — Discover available tools and their input schemas
- **`tools/call`** — Invoke a tool by name with arguments

The server URL follows the pattern:
```
https://{account}.snowflakecomputing.com/api/v2/databases/{db}/schemas/{schema}/mcp-servers/{name}
```

In [ ]:
#| export
class MCPError(Exception):
    """Raised when the MCP server returns a JSON-RPC error."""
    def __init__(self, code: int, message: str, data=None):
        self.code = code
        self.message = message
        self.data = data
        super().__init__(f"MCP error {code}: {message}")


class MCPClient:
    """JSON-RPC 2.0 client for Snowflake Managed MCP Servers."""

    def __init__(self, url: str, headers_factory: Callable[[], dict], timeout: int = 120):
        self.url = url
        self.headers_factory = headers_factory
        self.timeout = timeout
        self._req_counter = 0

    def request(self, method: str, params: dict | None = None) -> dict:
        """Send a JSON-RPC 2.0 request to the MCP server."""
        self._req_counter += 1
        body = {"jsonrpc": "2.0", "id": self._req_counter, "method": method}
        if params is not None:
            body["params"] = params
        resp = requests.post(
            self.url,
            headers=self.headers_factory(),
            json=body,
            timeout=self.timeout,
        )
        resp.raise_for_status()
        data = resp.json()
        err = data.get("error")
        if err:
            raise MCPError(err.get("code", -1), err.get("message", str(err)), err.get("data"))
        return data

    def tools_list(self) -> list[dict]:
        """List all tools available on the MCP server."""
        return self.request("tools/list").get("result", {}).get("tools", [])

    def call(self, tool_name: str, arguments: dict) -> dict:
        """Call an MCP tool by name with the given arguments."""
        return self.request("tools/call", {"name": tool_name, "arguments": arguments})


_default_client: MCPClient | None = None


def _get_default_client() -> MCPClient:
    """Lazily create the default MCP client on first use.

    The client is cached for the lifetime of the kernel/process.
    Call ``reset_default_mcp_client()`` to force re-creation.
    """
    global _default_client
    if _default_client is None:
        s = default_session()
        url = f"{s.host}/api/v2/databases/{s.database}/schemas/{s.mcp_schema}/mcp-servers/{s.mcp_server_name}"
        _default_client = MCPClient(url, get_headers)
    return _default_client


def reset_default_mcp_client() -> None:
    """Clear the cached default MCP client, forcing re-creation on next access."""
    global _default_client
    _default_client = None


def mcp_url() -> str:
    """Return the MCP server URL (lazily resolved from default session)."""
    return _get_default_client().url


def mcp_request(method: str, params: dict | None = None) -> dict:
    """Send a JSON-RPC 2.0 request using the default client."""
    return _get_default_client().request(method, params)


def mcp_tools_list() -> list[dict]:
    """List all tools using the default client."""
    return _get_default_client().tools_list()


def mcp_call(tool_name: str, arguments: dict) -> dict:
    """Call an MCP tool using the default client."""
    return _get_default_client().call(tool_name, arguments)


class MCPToolbox:
    """Ergonomic wrapper around an MCP server's tool inventory.

    Caches the tool list, provides discovery helpers, and pairs naturally
    with ``display_mcp_result()``. The primary happy path for MCP tool calling.

    Lower-level control is available via ``MCPClient`` and ``mcp_request()``.
    """

    def __init__(self, client: MCPClient | None = None):
        self._client = client
        self._tools: list[dict] | None = None

    @property
    def client(self) -> MCPClient:
        """The underlying MCPClient (default client if none was provided)."""
        return self._client or _get_default_client()

    @property
    def tools(self) -> list[dict]:
        """Cached tool list; fetched once on first access."""
        if self._tools is None:
            self._tools = self.client.tools_list()
        return self._tools

    @property
    def tool_names(self) -> list[str]:
        """List of tool names available on the server."""
        return [t["name"] for t in self.tools]

    def describe(self, tool_name: str) -> dict | None:
        """Return the full tool spec for a given tool name, or None if not found."""
        return next((t for t in self.tools if t["name"] == tool_name), None)

    def call(self, tool_name: str, arguments: dict) -> dict:
        """Call a tool by name. Returns the raw JSON-RPC response."""
        return self.client.call(tool_name, arguments)

    def refresh(self) -> "MCPToolbox":
        """Clear cached tool list, forcing re-fetch on next access. Returns self."""
        self._tools = None
        return self

    def __repr__(self) -> str:
        n = len(self._tools) if self._tools is not None else "?"
        return f"MCPToolbox(tools={n}, url={self.client.url!r})"

Let's discover what tools are available:

In [ ]:
#| eval: false
tools = mcp_tools_list()
for t in tools:
    print(f"  {t['name']:<30s} {t.get('description', '')[:60]}")

## Display Helpers

Rich display functions for rendering MCP responses in Jupyter notebooks with dark-mode compatible styling. `display_mcp_result` auto-detects the response type (SQL, Agent, Analyst, Search) and renders accordingly.

In [ ]:
#| export
def _render_error(error):
    from IPython.display import display, HTML
    msg = _html_escape(str(error.get("message", error)))
    display(HTML(
        f'<div style="padding:12px;background:#451a1a;border:1px solid #991b1b;'
        f'border-radius:8px;color:#fca5a5;"><b>Error:</b> {msg}</div>'
    ))


def _render_result_set(rs: dict, query_id: str | None = None):
    from IPython.display import display, HTML
    df = result_set_to_dataframe(rs)
    if query_id:
        display(HTML(f'<p style="color:#9ca3af;font-size:12px;">Query ID: {_html_escape(query_id)}</p>'))
    display(HTML(f'<p style="font-weight:600;color:#d1d5db;">Result ({len(df)} rows)</p>'))
    display(df.head(20))


def _render_sql(statement: str):
    from IPython.display import display, HTML
    display(HTML(
        f'<details><summary style="cursor:pointer;font-weight:600;color:#9ca3af;">SQL</summary>'
        f'<pre style="padding:12px;background:#1e293b;color:#e2e8f0;border-radius:8px;'
        f'font-size:13px;">{_html_escape(statement)}</pre></details>'
    ))


def _render_json(obj, max_chars: int = 3000):
    from IPython.display import display, HTML
    raw = json.dumps(obj, indent=2)
    text = _html_escape(raw[:max_chars])
    truncated = ' <i style="color:#9ca3af;">(truncated)</i>' if len(raw) > max_chars else ''
    display(HTML(
        f'<pre style="padding:12px;background:#1e293b;border:1px solid #334155;'
        f'border-radius:8px;font-size:13px;color:#e2e8f0;overflow-x:auto;max-height:400px;">'
        f'{text}{truncated}</pre>'
    ))


def _render_analyst_items(items: list):
    from IPython.display import display, Markdown
    for item in items:
        if not isinstance(item, dict):
            continue
        if "text" in item:
            display(Markdown(item["text"]))
        if "statement" in item:
            _render_sql(item["statement"])
        if "resultSet" in item:
            _render_result_set(item["resultSet"])


def display_mcp_result(response: dict, label: str = ""):
    """Best-effort notebook renderer for MCP tool responses.

    Attempts to auto-detect the response shape and render it appropriately:
    SQL (result_set), Agent (content), Analyst (list), Search (list of dicts),
    or GENERIC (raw JSON fallback). Not format-authoritative — treat as a
    convenience helper for interactive exploration.
    """
    from IPython.display import display, Markdown, HTML

    error = response.get("error")
    if error:
        _render_error(error)
        return

    if label:
        display(HTML(f'<h4 style="color:#60a5fa;margin:12px 0 4px;">{_html_escape(label)}</h4>'))

    text = extract_text(response)
    parsed = try_parse_json(text)

    if isinstance(parsed, dict):
        if "result_set" in parsed:
            _render_result_set(parsed["result_set"], parsed.get("query_id"))
        elif "content" in parsed:
            agent_data = parse_agent_response(response)
            if agent_data.get("answer"):
                display(Markdown(agent_data["answer"]))
            for rs in agent_data.get("result_sets", []):
                _render_result_set(rs)
        else:
            _render_json(parsed)
    elif isinstance(parsed, list):
        if parsed and isinstance(parsed[0], dict):
            has_analyst_keys = any(
                "text" in item or "statement" in item or "resultSet" in item
                for item in parsed if isinstance(item, dict)
            )
            if has_analyst_keys:
                _render_analyst_items(parsed)
            else:
                df = pd.DataFrame(parsed)
                display(df.head(20))
        else:
            _render_json(parsed)
    elif text:
        display(Markdown(text[:3000]))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()